<a href="https://colab.research.google.com/github/Joshi-kv/gen-ai/blob/main/VectorDB/pinecone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install langchain pinecone-client pypdf langchain-community langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [6]:
!pip install openai tiktoken langchain-pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.


In [12]:
import os

In [84]:
os.environ['PINECONE_API_KEY'] = ''
os.environ['OPENAI_API_KEY'] = ""

In [26]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI
from langchain_pinecone import PineconeVectorStore
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from openai import OpenAI as SyncOpenAIClient

In [2]:
!mkdir pdfs

In [4]:
!gdown 1EoRqNdZAiBBUneRIFffWAxa_3P87IsFo -O pdfs/offerletter.pdf

Downloading...
From: https://drive.google.com/uc?id=1EoRqNdZAiBBUneRIFffWAxa_3P87IsFo
To: /content/pdfs/offerletter.pdf
100% 444k/444k [00:00<00:00, 43.8MB/s]


In [5]:
!gdown 1OgLE6phbXE9HddRszyerUcHP-z1yq_ct -O pdfs/resume.pdf

Downloading...
From: https://drive.google.com/uc?id=1OgLE6phbXE9HddRszyerUcHP-z1yq_ct
To: /content/pdfs/resume.pdf
100% 33.1k/33.1k [00:00<00:00, 51.2MB/s]


Extract text from pdf

In [28]:
loader = PyPDFDirectoryLoader('pdfs')
documents = loader.load()

In [29]:
#convert to chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=10)
text_chunks = text_splitter.split_documents(documents)

In [10]:
text_chunks

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-01-10T04:30:04+00:00', 'author': '', 'keywords': '', 'moddate': '2026-01-10T04:30:04+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'pdfs/resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='JOSHI K V\nJunior Python Developer\nPhone: +91 9207429501\nEmail: joshikv3705@gmail.com\nLocation: Wayanad, Kerala, India\nLinkedIn: linkedin.com/in/joshi-kv-97a540228\nGitHub: github.com/Joshi-kv\nSUMMARY\nPython Developer with 3 years of experience designing and deploying scalable REST APIs\nand full-stack applications. Skilled in Django, FastAPI, React.js, and Next.js with hands-on\nexperience in PostgreSQL, MongoDB, and cloud deployments usingAWS. Strong foundation'),
 Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with h

In [50]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [51]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipython-input-914705057.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [56]:
from pinecone import Pinecone, ServerlessSpec

In [59]:
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))

In [65]:
index_name='my-hf-index'
pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)


{
    "name": "my-hf-index",
    "metric": "cosine",
    "host": "my-hf-index-5i8baod.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [66]:
doc_search = PineconeVectorStore.from_documents(text_chunks, embedding, index_name=index_name)

In [67]:
print(pc.list_indexes())

[{
    "name": "test",
    "metric": "cosine",
    "host": "test-5i8baod.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1024,
    "deletion_protection": "disabled",
    "tags": {
        "embedding_model": "text-embedding-3-large"
    }
}, {
    "name": "my-hf-index",
    "metric": "cosine",
    "host": "my-hf-index-5i8baod.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}]


In [68]:
docs = doc_search.similarity_search("hello", k=2)
print(docs)

[Document(id='939542e6-dc27-4c15-acb8-20a2f9db7d20', metadata={'creationdate': "20221122050936+00'00'", 'creator': 'PyPDF', 'moddate': "20221122050936+00'00'", 'page': 0.0, 'page_label': '1', 'producer': 'mPDF 6.1', 'source': 'pdfs/offerletter.pdf', 'title': 'Certificates', 'total_pages': 1.0}, page_content='22/11/2022\nOFFER LETTER\nHello Joshi Kv\nWe are delighted to offer you an internship with Inmakes Infotech Pvt. Ltd.  Get ready\nto start a journey that will give wings to your career dreams. You have been selected as a\nJr. Python Full Stack Intern for 3 Months, which will commence on 20-November-2022.\nThe internship will be carried out online on our dedicated platform www.inmakeslh.in\nSo, all the best for a fresh start in your career.\nBest Regards,\nNicemol P Surendran\nHuman Resource Manager'), Document(id='559cb58f-f67e-47e7-b4e2-87413a6247fe', metadata={'creationdate': "20221122050936+00'00'", 'creator': 'PyPDF', 'moddate': "20221122050936+00'00'", 'page': 0.0, 'page_label

In [69]:
doc_search = PineconeVectorStore.from_documents(
    text_chunks,
    embedding,
    index_name="my-hf-index",
    namespace="docs"
)

In [70]:
doc_search = PineconeVectorStore.from_documents(
    text_chunks,
    embedding,
    index_name="my-hf-index",
    ids=[str(i) for i in range(len(text_chunks))]
)

In [73]:
doc_search = PineconeVectorStore.from_existing_index(index_name, embedding)

In [74]:
doc_search.similarity_search("Describe about Joshi", k=3)

[Document(id='7c35e808-060d-4a7e-ae85-ef0b322c9012', metadata={'author': '', 'creationdate': '2026-01-10T04:30:04+00:00', 'creator': 'LaTeX with hyperref', 'keywords': '', 'moddate': '2026-01-10T04:30:04+00:00', 'page': 0.0, 'page_label': '1', 'producer': 'pdfTeX-1.40.26', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'source': 'pdfs/resume.pdf', 'subject': '', 'title': '', 'total_pages': 2.0, 'trapped': '/False'}, page_content='JOSHI K V\nJunior Python Developer\nPhone: +91 9207429501\nEmail: joshikv3705@gmail.com\nLocation: Wayanad, Kerala, India\nLinkedIn: linkedin.com/in/joshi-kv-97a540228\nGitHub: github.com/Joshi-kv\nSUMMARY\nPython Developer with 3 years of experience designing and deploying scalable REST APIs\nand full-stack applications. Skilled in Django, FastAPI, React.js, and Next.js with hands-on\nexperience in PostgreSQL, MongoDB, and cloud deployments usingAWS. Strong foundation'),
 Document(id='0', metadata=

In [75]:
retriever = doc_search.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [76]:
from langchain_openai import ChatOpenAI

In [77]:
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    model="mistralai/mistral-7b-instruct",
    temperature=0
)

In [83]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# your retriever already created earlier
# retriever = vectorbd.as_retriever(search_kwargs={'k':2})

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"), # Corrected API key environment variable
    base_url="https://openrouter.ai/api/v1"
)

prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.
If you don't know, say you don't know.

Context:
{context}

Question:
{question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

response = rag_chain.invoke("Describe about Joshi")
print(response)

Joshi K V is a Junior Python Developer with 3 years of experience in designing and deploying scalable REST APIs and full-stack applications. He is skilled in Django, FastAPI, React.js, and Next.js, and has hands-on experience with PostgreSQL, MongoDB, and cloud deployments using AWS. Joshi is located in Wayanad, Kerala, India, and can be contacted via phone or email. He also has professional profiles on LinkedIn and GitHub. Additionally, he received an internship offer from Inmakes Infotech Pvt. Ltd. as a Junior Python Full Stack Intern, which commenced on November 20, 2022.
